# GUI API

This notebook shows practical calls to the Flask endpoints used by the web UI.

Before running cells, start the app in a terminal:

`python run.py serve`


In [1]:
import requests
from pprint import pprint

BASE_URL = "http://127.0.0.1:5000"

def check_server(url: str) -> None:
    try:
        resp = requests.get(f"{url}/api/stats", timeout=3)
        resp.raise_for_status()
    except Exception as exc:
        raise RuntimeError(
            "Flask server is not running. Start it in a terminal with: python run.py serve"
        ) from exc

check_server(BASE_URL)
print(f"Server reachable at {BASE_URL}")

Server reachable at http://127.0.0.1:5000


## 1) Get Current Stats

In [2]:
resp = requests.get(f"{BASE_URL}/api/stats", timeout=30)
resp.raise_for_status()
pprint(resp.json())


{'categories': ['astro-ph.CO', 'astro-ph.GA', 'astro-ph.HE', 'astro-ph.SR'],
 'data_dir': 'data',
 'database': {'thumbs_down': 0,
              'thumbs_up': 1,
              'total_papers': 56,
              'total_rated': 1,
              'with_embeddings': 56},
 'model': {'embedding_dim': 384,
           'learning_rate': 0.001,
           'model_path': 'data/preference_model.pt',
           'parameters': 59649,
           'recent_losses': [],
           'total_trained': 2}}


## 2) Fetch New Papers

In [3]:
payload = {"max_results": 20, "days_back": 2, "with_summaries": False}
resp = requests.post(f"{BASE_URL}/api/fetch", json=payload, timeout=120)
resp.raise_for_status()
pprint(resp.json())


{'new_papers': 0, 'status': 'ok'}


## 3) Rate a Paper
Use an arXiv id that exists in your DB. Rating: `1` for like, `0` for dislike.

In [4]:
paper_id = "2401.12345"  # replace with a real id from your database
payload = {"arxiv_id": paper_id, "rating": 1}
resp = requests.post(f"{BASE_URL}/api/rate", json=payload, timeout=30)
print(resp.status_code)
pprint(resp.json())


200
{'reason': 'no embedding', 'status': 'rated', 'trained': False}


## 4) Run Summary Generation

In [5]:
payload = {"limit": 10, "only_missing": True}
resp = requests.post(f"{BASE_URL}/api/summarize", json=payload, timeout=120)
resp.raise_for_status()
pprint(resp.json())


{'processed': 0, 'status': 'no_work', 'updated': 0}


## 5) Retrain the Preference Model

In [6]:
payload = {"epochs": 20}
resp = requests.post(f"{BASE_URL}/api/retrain", json=payload, timeout=120)
resp.raise_for_status()
pprint(resp.json())


{'message': 'No rated papers with embeddings found', 'status': 'no_data'}
